# SmolVLA v2 ファインチューニング：牡蠣人形の言語条件付け（2カメラ・89本）

**v1(60本1カメラ)は0/5で言語切替失敗** → v2は**手首カメラ追加＋89本**で再挑戦。
- `pick up the white oyster plush and put it in the box`（箱, 30本）
- `pick up the white oyster plush and make it bow`（お辞儀, 29本）
- `pick up the green tape and put it in the box`（緑テープ→箱, 30本）

計89エピソード・2カメラ(俯瞰+手首)。

**約2.5時間で終わる設定（20,000ステップ）。A100必須。**

**事前準備**: `oyster_v2_dataset.zip` を Googleドライブ `MyDrive/so101/` にアップロード

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ZIP = "/content/drive/MyDrive/so101/oyster_v2_dataset.zip"

!mkdir -p /content/dataset
!unzip -q "$DRIVE_ZIP" -d /content/dataset
!ls /content/dataset

In [ ]:
!pip install -q "lerobot[dataset,training]==0.6.0" accelerate peft num2words "torchao>=0.16.0"

In [ ]:
import glob

candidates = glob.glob("/content/dataset/so101_oyster_v2_2cam_*")
assert len(candidates) == 1, f"データセットフォルダが1つに絞れません: {candidates}"
DATASET_ROOT = candidates[0]
print("DATASET_ROOT =", DATASET_ROOT)

EPISODES = "[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88]"  # 全89エピソード

## 疎通確認（任意・T4でOK、3000ステップ）
短時間で回してエラーが無いか先に確認したい場合のみ。不要なら飛ばして本番へ。

In [ ]:
!lerobot-train \
  --dataset.repo_id=local/so101_oyster_v2_2cam \
  --dataset.root="$DATASET_ROOT" \
  --dataset.episodes="$EPISODES" \
  --dataset.video_backend=pyav \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.load_vlm_weights=true \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --peft.r=16 \
  --output_dir=/content/drive/MyDrive/so101/train_output/smolvla_oyster_v2_short \
  --job_name=smolvla_oyster_v2_short \
  --steps=3000 \
  --batch_size=8 \
  --save_freq=3000 \
  --log_freq=50 \
  --wandb.enable=false

## ★本番（A100・20,000ステップ≒2.5時間）

「ランタイム→ランタイムのタイプを変更」で**A100**へ。切替でリセットされるので上のセルから順に再実行。

- `--steps=20000`：2カメラで1ステップが重いのでv1の50kより少なめ設定。約2.5時間で完走見込み。
- `--save_freq=2500`：2500ステップごとにチェックポイント保存。2.5hで止まっても直近が使える。
- 出力は Drive の `train_output/smolvla_oyster_v2_long/checkpoints/` に保存。

In [ ]:
!lerobot-train \
  --dataset.repo_id=local/so101_oyster_v2_2cam \
  --dataset.root="$DATASET_ROOT" \
  --dataset.episodes="$EPISODES" \
  --dataset.video_backend=pyav \
  --policy.type=smolvla \
  --policy.pretrained_path=lerobot/smolvla_base \
  --policy.load_vlm_weights=true \
  --policy.device=cuda \
  --policy.push_to_hub=false \
  --peft.r=16 \
  --output_dir=/content/drive/MyDrive/so101/train_output/smolvla_oyster_v2_long \
  --job_name=smolvla_oyster_v2_long \
  --steps=20000 \
  --batch_size=8 \
  --save_freq=2500 \
  --log_freq=100 \
  --wandb.enable=false

In [ ]:
!ls -la /content/drive/MyDrive/so101/train_output/smolvla_oyster_v2_long/checkpoints